In [1]:
import torch, os

## 1. image Embedding

In [2]:
from tools.embedding import encode_directory, ImageEncoderFeatureExtractor

data_dir = "..\\1_dataset\\data_"
embs_dir = "..\\2_embeddings\\data_"
data_type = ["syn", "real"]

encs_set = {
    "clip":         ["openai/clip-vit-base-patch16", 512],
    "resnet":       ["microsoft/resnet-50", 1024],
    "efficientnet": ["google/efficientnet-b7", 32],
    "sam":          ["facebook/sam-vit-base",8],
    "sam2":         ["facebook/sam2.1-hiera-base-plus",8],
}

In [3]:
for i in data_type:
    for enc, (ckpt, bs) in encs_set.items():
        encode_directory(enc, data_dir + i, embs_dir + i,
                         checkpoint=ckpt, batch_size=bs)       

In [3]:
# applications
from tools.embedding import encode_directory, ImageEncoderFeatureExtractor

data_dir = "F:\\2_matchwatch_dataSynGen\\deepFashion\\selected_images"
# data_dir = "F:\\lsp_images"
embs_dir = "..\\2_embeddings_applications\\deepFashion"

encode_directory("clip", data_dir, embs_dir, checkpoint="openai/clip-vit-base-patch16", batch_size=512)           

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 3841.63it/s]


[START] clip :
[FINISH] 12231 embeddings
  L spatial : (12231, 768, 14, 14)
  L pooled : (12231, 512)




**local feature map(l_fmp))**
- preserved spatial features
- aka) local feaures, dense feature map

**global embedding(g_emb)**
- vector that represent the image
- aka) image embedding. global descriptor  
<br>

| encoder | g_emb | l_fmp |
| ----------- | ----------- | ----------- |
| CLIP | projected image embedding | patch (token) embeddings, patch features |
| dinov2 | CLS token embedding | patch tokens, patch features |
| resnet | pooled features | feature map |
| efficientnet | pooled features | feature map |
| sam | None | image embedding|
| sam2 | None | image embedding, FPN feature |

Input → Backbone (CNN / ViT) → Feature map / tokens   ← LOCAL  (keeps spatial info)<br>
↓<br>
Pooling / CLS / Projection<br>
↓<br>
Single vector           ← GLOBAL (image-level)<br>

| Encoder | l_fmp `(B,C,H,W)` | g_emb `(B,D)` | How g_emb is formed | Global ⟵ Local? |
|---|---|---|---|---|
| **CLIP** ViT-B/16 | `(B,768,14,14)` | `(B,512)` | CLS token → **visual projection** | ❌ no |
| **ResNet-50** | `(B,2048,7,7)` | `(B,2048)` | Global Avg Pooling of feature map | ✅ yes |
| **EfficientNet-B7** | `(B,2560,19,19)` | `(B,2560)` | Global Avg Pooling of feature map | ✅ yes |
| **EfficientNet-B0** | `(B,1280,7,7)` | `(B,1280)` | Global Avg Pooling of feature map | ✅ yes |
| **DINOv2** ViT-B | `(B,768,16,16)` | `(B,768)` | CLS token | ❌ no |
| **SAM** ViT-B | `(B,256,64,64)` | — (None) | no global by design | — |
| **SAM2** Hiera-B+ | `(B,256,64,64)` | — (None) | no global by design | — |

### Notes by family
- **CNN (ResNet, EfficientNet):** global = **GAP of the local feature map**, so global is a true summary of local and can be recomputed as `local.mean(dim=[2,3])`.
- **ViT (CLIP, DINOv2):** local = **patch tokens** (CLS excluded, reshaped to a grid); global = **CLS token**. CLS is *not* a pooling of the patches — it is a sibling token updated via attention. **CLIP** additionally passes CLS through a **projection layer** into the 512-D image–text space, so its global is text-aligned and even further from local.
- **Segmentation encoders (SAM, SAM2):** produce only a **dense image embedding** (local); no native global vector, since a mask decoder consumes the spatial map directly.

> EfficientNet dims depend on the variant: **B0** → `1280 / 7×7`, **B7** → `2560 / 19×19` (600px input).

> SAM2 local is the coarsest FPN scale `(B,256,64,64)`; full FPN also yields `(B,32,256,256)` and `(B,64,128,128)`.

## 2. train model

`train.py` 를 이용한 학습.

전체 매트릭스(enc × path × cond × P × seed)는 무거우므로 **(1) 스모크 테스트로 파이프라인 확인 → (2) 부분/전체 실험** 순으로 진행한다.

- **path** : `mEnc`(local/patch 입력) / `MLP`(global/pooled 입력, SAM·SAM2 제외)
- **cond** : A(random) / B(aligned+distinct → hard negative)
- **P** : 배치 내 구도 수 스윕 {256,128,64,32}
- **seed** : 5회 반복 → 결과는 mean ± std

In [4]:
# from train import run_experiments

# results = run_experiments()

In [5]:
# # 293m 6s 소요 -- 0819
# # 1234m 18s 소요 -- 0821
# for i in results:
#     print(f"Enc : {i['setting']}")
#     print(f"test_syn's mAP : {i['test_syn']['mAP']}")
#     print(f"test_real's mAP : {i['test_real']['mAP']}", end='\n\n')

In [6]:
# results

In [7]:
from train import run_experiments
from time import sleep

results_sam2 = run_experiments("..\\2_embeddings\\data_syn", "..\\2_embeddings\\data_real",["sam2"], [256, 128, 64, 32, 16, 8, 4, 2], [0,1,2,3,4], 1000000, 100, 1e-4, 0.05, 42,[0.6, 0.2, 0.2],0.6,True)
sleep(300)
print('\n-----------------------------------------------------------------')

results_sam = run_experiments("..\\2_embeddings\\data_syn", "..\\2_embeddings\\data_real",["sam"], [256, 128, 64, 32, 16, 8, 4, 2], [0,1,2,3,4], 1000000, 100, 1e-4, 0.05, 42,[0.6, 0.2, 0.2],0.6,True)
sleep(300)
print('\n-----------------------------------------------------------------')

results_efficientnet = run_experiments("..\\2_embeddings\\data_syn", "..\\2_embeddings\\data_real",["efficientnet"], [256, 128, 64, 32, 16, 8, 4, 2], [0,1,2,3,4], 1000000, 100, 1e-4, 0.05, 42,[0.6, 0.2, 0.2],0.6,True)
sleep(300)
print('\n-----------------------------------------------------------------')

results_clip = run_experiments("..\\2_embeddings\\data_syn", "..\\2_embeddings\\data_real",["clip"], [256, 128, 64, 32, 16, 8, 4, 2], [0,1,2,3,4], 1000000, 100, 1e-4, 0.05, 42,[0.6, 0.2, 0.2],0.6,True)
sleep(300)
print('\n-----------------------------------------------------------------')

results_resnet = run_experiments("..\\2_embeddings\\data_syn", "..\\2_embeddings\\data_real",["resnet"], [256, 128, 64, 32, 16, 8, 4, 2], [0,1,2,3,4], 1000000, 100, 1e-4, 0.05, 42,[0.6, 0.2, 0.2],0.6,True)

c:\Users\rcjh5957\Downloads\matchwatch_0713\3_code\train.py:53: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp)
c:\Users\rcjh5957\Downloads\matchwatch_0713\3_code\train.py:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[sam2_256_0] : step   100 / loss 0.6667 / val_mAP 0.8289
[sam2_256_0] : step   200 / loss 0.1544 / val_mAP 0.8510
[sam2_256_0] : step   300 / loss 0.0504 / val_mAP 0.8582
[sam2_256_0] : step   400 / loss 0.0298 / val_mAP 0.8619
[sam2_256_0] : step   500 / loss 0.0151 / val_mAP 0.8643
[sam2_256_0] : step   600 / loss 0.0127 / val_mAP 0.8669
[sam2_256_0] : step   700 / loss 0.0099 / val_mAP 0.8679
[sam2_256_0] : step   800 / loss 0.0084 / val_mAP 0.8694
[sam2_256_0] : step   900 / loss 0.0070 / val_mAP 0.8702
[sam2_256_0] : step  1000 / loss 0.0061 / val_mAP 0.8713
  --> real mAP : 0.7717644749662984
[sam2_256_0] : step  1100 / loss 0.0051 / val_mAP 0.8720
[sam2_256_0] : step  1200 / loss 0.0044 / val_mAP 0.8730
[sam2_256_0] : step  1300 / loss 0.0038 / val_mAP 0.8739
[sam2_256_0] : step  1400 / loss 0.0036 / val_mAP 0.8745
[sam2_256_0] : step  1500 / loss 0.0030 / val_mAP 0.8748
[sam2_256_0] : step  1600 / loss 0.0032 / val_mAP 0.8756
[sam2_256_0] : step  1700 / loss 0.0025 / val_mAP 0.

In [18]:
# applications
from train import run_experiments
results_sam2 = run_experiments("..\\2_embeddings\\data_syn", "..\\2_embeddings_applications\\deepFashion",["sam2"], [16], [4], 1000000, 100, 1e-4, 0.05, 42,[0.6, 0.2, 0.2],0.6,True)

c:\Users\rcjh5957\Downloads\matchwatch_0713\3_code\train.py:53: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp)
c:\Users\rcjh5957\Downloads\matchwatch_0713\3_code\train.py:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


[sam2_16_4] : step   100 / loss 1.7375 / val_mAP 0.4993
[sam2_16_4] : step   200 / loss 0.2614 / val_mAP 0.7043
[sam2_16_4] : step   300 / loss 0.2509 / val_mAP 0.7597
[sam2_16_4] : step   400 / loss 0.3103 / val_mAP 0.7844
[sam2_16_4] : step   500 / loss 0.2186 / val_mAP 0.7970
[sam2_16_4] : step   600 / loss 0.1095 / val_mAP 0.8058
[sam2_16_4] : step   700 / loss 0.0214 / val_mAP 0.8184
[sam2_16_4] : step   800 / loss 0.0966 / val_mAP 0.8241
[sam2_16_4] : step   900 / loss 0.2151 / val_mAP 0.8279
[sam2_16_4] : step  1000 / loss 0.0090 / val_mAP 0.8325
[sam2_16_4] : step  1100 / loss 0.0189 / val_mAP 0.8361
[sam2_16_4] : step  1200 / loss 0.0073 / val_mAP 0.8381
[sam2_16_4] : step  1300 / loss 0.0647 / val_mAP 0.8386
[sam2_16_4] : step  1400 / loss 0.0011 / val_mAP 0.8430
[sam2_16_4] : step  1500 / loss 0.0179 / val_mAP 0.8492
[sam2_16_4] : step  1600 / loss 0.0226 / val_mAP 0.8511
[sam2_16_4] : step  1700 / loss 0.0009 / val_mAP 0.8545
[sam2_16_4] : step  1800 / loss 0.0072 / val_mAP

In [ ]:
for query, answers in results_sam2[0]['test_real']['retrievals'].items():
    if answers[1][0] > 0.9:
        print(query)
        print(answers[0])
        print(answers[1])
        print()

In [ ]:
results_sam2[0]['test_real']['retrievals']

In [ ]:
import json
results_sam2_dict = results_sam2[0]['test_real']['retrievals']

with open("applications.json", "w", encoding="utf-8") as f:
  json.dump(results_sam2_dict, f, ensure_ascii=False, indent=4)

In [8]:
for result in [results_sam2, results_sam, results_clip, results_efficientnet]:
    for i in result:
        print(f"Enc : {i['setting']}")
        print(f"test_syn's mAP : {i['test_syn']['mAP']}")
        print(f"test_real's mAP : {i['test_real']['mAP']}", end='\n\n')
    print('\n==========================================\n')

Enc : sam2_256_0
test_syn's mAP : 0.8229354633928077
test_real's mAP : 0.7909624297081144

Enc : sam2_128_0
test_syn's mAP : 0.814496708451083
test_real's mAP : 0.807425283984312

Enc : sam2_64_0
test_syn's mAP : 0.8217351375556647
test_real's mAP : 0.8112787017906357

Enc : sam2_32_0
test_syn's mAP : 0.7685950014288891
test_real's mAP : 0.7720002621685234

Enc : sam2_16_0
test_syn's mAP : 0.791887789167925
test_real's mAP : 0.8206768943581396

Enc : sam2_8_0
test_syn's mAP : 0.7672373481900607
test_real's mAP : 0.783153470478639

Enc : sam2_4_0
test_syn's mAP : 0.7060921732095176
test_real's mAP : 0.7864501611268915

Enc : sam2_2_0
test_syn's mAP : 0.42053563131102806
test_real's mAP : 0.5617715757026114

Enc : sam2_256_1
test_syn's mAP : 0.8232239619791578
test_real's mAP : 0.7888322964342505

Enc : sam2_128_1
test_syn's mAP : 0.8291237475340526
test_real's mAP : 0.8193679671037901

Enc : sam2_64_1
test_syn's mAP : 0.793687641922441
test_real's mAP : 0.7918642126237382

Enc : sam2_32

In [9]:
results_dict = {
    "results_sam2": results_sam2,
    "results_sam": results_sam,
    "results_efficientnet": results_efficientnet,
    "results_clip": results_clip,
    "results_resnet": results_resnet,
}

for name, data in results_dict.items():
    with open(f"{name}_output.txt", "w", encoding="utf-8") as f:
        f.write(str(data))
        print(f"SAVE : {name}")

SAVE : results_sam2
SAVE : results_sam
SAVE : results_efficientnet
SAVE : results_clip
SAVE : results_resnet


## 3. evaluation

학습 결과 분석 : **(1) seed 평균±편차 요약표**, **(2) 검증 mAP 곡선(seed 밴드)**.

`train.py` 가 각 (enc, path, cond, P) 에 대해 최종 test(seen/unseen × HARD/EASY)를 이미 평가하므로, 여기서는 집계·시각화만 수행한다.
- **HARD** : 쿼리와 bg 같거나 ch 같은 후보를 갤러리에서 제외(배경/캐릭터 지름길 차단)
- **EASY** : 자기 자신만 제외
- **seen** : 주평가(6×6) / **unseen** : 일반화(test 구도 × 4×4)